# Phase 7b: DistilBERT fine-tune on Colab

Thin wrapper: clones the repo, installs deps, trains DistilBERT on the
weak-labelled train set, scores the 200-row gold eval, saves predictions
+ weights to Drive. No logic lives here — `trustlayer.classify.bert`
owns everything (smoke gate → 3-epoch train → predict).

Use a GPU runtime (T4): Runtime → Change runtime type → T4 GPU.

Before running: upload `train.jsonl` + `eval.jsonl` when the file picker
appears (≈11 MB + 0.2 MB, from your local `data/classify/`). The run
takes ~10–20 min on T4. Output in Drive at `trustlayer/distilbert/`:
`distilbert.json` (predictions, ~100 KB — download this) + `model/`
(fine-tuned weights, ~260 MB — download if you want local re-predict).
The guide in `docs/colab-phase7.md` has the exact download paths.

In [ ]:
from google.colab import drive, files
drive.mount('/content/drive')

!git clone https://github.com/ctru0009/trustlayer.git /content/trustlayer
%cd /content/trustlayer/python
!pip install -q uv
!uv sync --extra embed

In [ ]:
# Upload train.jsonl + eval.jsonl from your local data/classify/
# (file picker; ~11 MB total). They land in the notebook's cwd.
import shutil
from pathlib import Path

uploaded = files.upload()
DATA = Path('/content/trustlayer/data/classify')
DATA.mkdir(parents=True, exist_ok=True)
for name in ('train.jsonl', 'eval.jsonl'):
    if name not in uploaded:
        raise SystemExit(f'missing upload: {name} — re-run and pick both files')
    shutil.move(name, DATA / name)
    print(f'[data] staged {name}')

In [ ]:
# Smoke (50 steps, must show dropping loss) → train (3 epochs) →
# predict (200 gold rows). One command; interrupt and re-run resumes
# nothing (training restarts) so let it finish (~10–20 min on T4).
!uv run --extra embed python -m trustlayer.classify.bert --mode all --device cuda

In [ ]:
import shutil
from pathlib import Path

DATA = Path('/content/trustlayer/data/classify')
BACKUP = Path('/content/drive/MyDrive/trustlayer/distilbert')
BACKUP.mkdir(parents=True, exist_ok=True)

pred = DATA / 'results' / 'distilbert.json'
if not pred.exists():
    raise SystemExit('predictions missing — did the train cell finish?')
shutil.copy(pred, BACKUP / pred.name)
print(f'[drive] saved {pred.name} ({pred.stat().st_size // 1024} KB)')

# Weights are large (~260 MB); back up for reproducibility.
model_dir = DATA / 'distilbert'
if model_dir.is_dir():
    shutil.rmtree(BACKUP / 'model', ignore_errors=True)
    shutil.copytree(model_dir, BACKUP / 'model')
    print('[drive] saved model/')